# ICD 341 – Advanced Statistical Computing: Labs 5–8
**Name:** Frazer  &nbsp;&nbsp; **Student number:** YOUR_STUDENT_NUMBER

In [ ]:
import numpy as np
import scipy.linalg as la
import scipy.sparse as sp
from scipy import optimize, stats
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# LAB 5 – Linear systems and regression

## Q1. Solve 3x + 2y = 12, x + 4y = 10

In [ ]:
A = np.array([[3, 2],
              [1, 4]], dtype=float)
b = np.array([12, 10], dtype=float)

sol = np.linalg.solve(A, b)
print("x =", sol[0], ", y =", sol[1])
print("Check A @ sol =", A @ sol)   # should be [12, 10]

**Answer:** x = 2.8, y = 1.8 (check: 3(2.8)+2(1.8) = 12 and 2.8+4(1.8) = 10).

## Q2. A 3 × 3 system with `np.linalg.solve()`

In [ ]:
A3 = np.array([[2, 1, -1],
               [-3, -1, 2],
               [-2, 1, 2]], dtype=float)
b3 = np.array([8, -11, -3], dtype=float)

x3 = np.linalg.solve(A3, b3)
print("Solution:", x3)
print("Residual A@x - b:", A3 @ x3 - b3)

## Q3. Why `np.linalg.solve(A, b)` is preferred to `inv(A) @ b`
1. **Speed:** `solve` uses LU factorisation (about 2n³/3 operations) and goes straight to the solution. Forming A⁻¹ costs about 2n³ operations, and then you still have to multiply by b.
2. **Numerical accuracy:** computing an explicit inverse amplifies rounding errors, especially when A is ill-conditioned. `solve` gives a smaller residual.
3. **Memory / structure:** the inverse of a sparse or structured matrix is usually dense, so it wastes memory; factorisation avoids this.
4. **Intent:** we only need the vector x = A⁻¹b, not the whole inverse matrix.

In [ ]:
# Small demonstration on an ill-conditioned (Hilbert) matrix
n = 12
H = la.hilbert(n)
x_true = np.ones(n)
bH = H @ x_true

x_solve = np.linalg.solve(H, bH)
x_inv   = np.linalg.inv(H) @ bH
print("Error using solve:", np.linalg.norm(x_solve - x_true))
print("Error using inv  :", np.linalg.norm(x_inv - x_true))

## Q4. Regression of consumption (Y) on disposable income (X)

In [ ]:
X_inc = np.array([30, 45, 60, 100, 150, 200, 240, 250, 300, 320], dtype=float)
Y_con = np.array([25, 20, 40, 65, 120, 190, 200, 200, 260, 220], dtype=float)

# (i) Design matrix: a column of ones (intercept) and the income column
X = np.column_stack([np.ones_like(X_inc), X_inc])
print("Design matrix X:\n", X)

# Normal equations: (X'X) beta = X'y
beta = np.linalg.solve(X.T @ X, X.T @ Y_con)
print("\nbeta0 (intercept) =", beta[0])
print("beta1 (slope)     =", beta[1])

# Cross-check with least squares
print("lstsq check:", np.linalg.lstsq(X, Y_con, rcond=None)[0])

In [ ]:
# (ii) Predicted consumption when income = 181
x_new = 181
y_hat = beta[0] + beta[1] * x_new
print(f"Estimated consumption at income {x_new}: {y_hat:.4f}")

plt.scatter(X_inc, Y_con, label="Data")
xs = np.linspace(X_inc.min(), X_inc.max(), 100)
plt.plot(xs, beta[0] + beta[1]*xs, color="red", label="Fitted line")
plt.scatter([x_new], [y_hat], color="green", zorder=5, label="Prediction at 181")
plt.xlabel("Disposable income X"); plt.ylabel("Consumption Y"); plt.legend()
plt.show()

# LAB 6 – Matrix decompositions and sparse matrices

## Q1. Covariance matrix and Cholesky decomposition

In [ ]:
rng = np.random.default_rng(341)
data = rng.normal(size=(100, 3))
data[:, 1] += 0.5 * data[:, 0]
data[:, 2] += 0.3 * data[:, 1]

S = np.cov(data, rowvar=False)      # 3x3 covariance matrix
print("Covariance matrix S:\n", S)

L = np.linalg.cholesky(S)           # S = L L^T, L lower triangular
print("\nL:\n", L)
print("\nL @ L.T equals S?", np.allclose(L @ L.T, S))

## Q2. QR decomposition

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 10]], dtype=float)
Q, R = np.linalg.qr(M)
print("Q:\n", Q)
print("R:\n", R)
print("Q @ R equals M?     ", np.allclose(Q @ R, M))
print("Q orthogonal (Q'Q=I)?", np.allclose(Q.T @ Q, np.eye(3)))

## Q3. Reconstruct a matrix using SVD

In [ ]:
U, s, Vt = np.linalg.svd(M)
print("Singular values:", s)

M_rebuilt = U @ np.diag(s) @ Vt
print("Reconstructed:\n", M_rebuilt)
print("Matches original?", np.allclose(M, M_rebuilt))

# Rank-1 approximation (keeps only the biggest singular value)
M1 = s[0] * np.outer(U[:, 0], Vt[0, :])
print("\nRank-1 approximation:\n", M1)

## Q4. Condition number

In [ ]:
print("cond(M)        =", np.linalg.cond(M))
print("cond(Identity) =", np.linalg.cond(np.eye(3)))
print("cond(Hilbert 8)=", np.linalg.cond(la.hilbert(8)))
# Same as the ratio of largest to smallest singular value:
print("s_max / s_min  =", s[0] / s[-1])

A condition number near 1 means a well-conditioned problem. A very large value (like the Hilbert matrix) means the matrix is nearly singular and solutions are very sensitive to small errors.

## Q5. A sparse 1000 × 1000 matrix

In [ ]:
S_sparse = sp.random(1000, 1000, density=0.01, format="csr", random_state=1)
print("Shape:", S_sparse.shape)
print("Non-zero entries:", S_sparse.nnz, "out of", 1000*1000)

dense = S_sparse.toarray()
sparse_bytes = S_sparse.data.nbytes + S_sparse.indices.nbytes + S_sparse.indptr.nbytes
print("Dense memory  :", dense.nbytes / 1e6, "MB")
print("Sparse memory :", sparse_bytes / 1e6, "MB")

## Q6. Why sparse matrices are useful for large datasets
- **Memory:** only the non-zero values and their positions are stored, so a matrix that is mostly zeros takes a tiny fraction of the dense size (see the output above).
- **Speed:** operations such as multiplication loop over non-zero entries only, so they are much faster.
- **Scalability:** matrices that would not fit in RAM when dense (e.g. 1,000,000 × 1,000,000) can still be handled.
- **Typical uses:** text data (document-term matrices), graphs/networks, recommender systems, finite-element and large regression problems.

# LAB 7 – Optimisation, MLE and logistic regression

## Q1. Gradient descent for f(x) = (x − 10)²

In [ ]:
def f(x):      return (x - 10) ** 2
def grad_f(x): return 2 * (x - 10)

def gradient_descent(grad, x0, lr=0.1, n_iter=100):
    x = x0
    path = [x]
    for _ in range(n_iter):
        x = x - lr * grad(x)
        path.append(x)
    return x, path

x_min, path = gradient_descent(grad_f, x0=0.0, lr=0.1, n_iter=100)
print("Minimiser found:", x_min, "  f(x) =", f(x_min))   # true minimum x = 10

plt.plot(path); plt.axhline(10, color="red", ls="--")
plt.xlabel("Iteration"); plt.ylabel("x"); plt.title("Gradient descent path")
plt.show()

## Q2. Compare BFGS and CG on f(x, y) = (x − 4)² + (y − 6)²

In [ ]:
def g(p):      return (p[0] - 4) ** 2 + (p[1] - 6) ** 2
def grad_g(p): return np.array([2 * (p[0] - 4), 2 * (p[1] - 6)])

x0 = np.array([0.0, 0.0])
for method in ["BFGS", "CG"]:
    res = optimize.minimize(g, x0, jac=grad_g, method=method)
    print(f"{method:5s} -> x = {res.x},  f = {res.fun:.2e},  "
          f"iterations = {res.nit},  f-evals = {res.nfev},  grad-evals = {res.njev}")

Both methods find the minimum at (4, 6). On this simple, perfectly quadratic function they perform almost identically. BFGS builds an approximation of the Hessian (more memory, fast convergence), while CG only keeps a few vectors, which makes it better for very large problems.

## Q3. Effect of changing the learning rate

In [ ]:
for lr in [0.01, 0.1, 0.5, 0.9, 1.0, 1.05]:
    x_end, p = gradient_descent(grad_f, x0=0.0, lr=lr, n_iter=30)
    print(f"lr = {lr:<5} -> x after 30 steps = {x_end:12.4f}   |x-10| = {abs(x_end-10):.3e}")

plt.figure(figsize=(7, 4))
for lr in [0.01, 0.1, 0.9, 1.05]:
    _, p = gradient_descent(grad_f, 0.0, lr=lr, n_iter=30)
    plt.plot(p, label=f"lr={lr}")
plt.ylim(-5, 25); plt.axhline(10, color="k", ls=":")
plt.xlabel("Iteration"); plt.ylabel("x"); plt.legend(); plt.show()

**Explanation:**
- **Too small** (0.01): steady but very slow convergence, so many iterations are needed.
- **Moderate** (0.1 – 0.5): fast, smooth convergence to the minimum x = 10.
- **Large but < 1** (0.9): overshoots and oscillates around the minimum, but still converges.
- **lr = 1.0:** each step jumps exactly to the other side of the minimum (0 → 20 → 0 → ...), so it bounces forever and never converges.
- **Too large** (> 1 here): the steps overshoot more each time, so the iterates diverge.

The learning rate controls the step size. It must balance speed against stability.

## Q4. MLE of the mean of normal observations with SciPy

In [ ]:
rng = np.random.default_rng(1)
obs = rng.normal(loc=5, scale=2, size=200)     # true mean = 5, sd = 2 (sd treated as known)

def neg_loglik(mu):
    return -np.sum(stats.norm.logpdf(obs, loc=mu, scale=2))

res = optimize.minimize_scalar(neg_loglik)
print("MLE of mean (optimiser):", res.x)
print("Sample mean            :", obs.mean())     # analytic MLE

In [ ]:
# Estimating mean and standard deviation together
def neg_loglik2(params):
    mu, log_sigma = params
    return -np.sum(stats.norm.logpdf(obs, loc=mu, scale=np.exp(log_sigma)))

res2 = optimize.minimize(neg_loglik2, x0=[0.0, 0.0], method="BFGS")
print("mu_hat =", res2.x[0], "  sigma_hat =", np.exp(res2.x[1]))

## Q5. Sigmoid function

In [ ]:
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-z))

print(sigmoid(np.array([-5, -1, 0, 1, 5])))

z = np.linspace(-8, 8, 200)
plt.plot(z, sigmoid(z)); plt.axhline(0.5, color="gray", ls=":")
plt.xlabel("z"); plt.ylabel("sigmoid(z)"); plt.title("Sigmoid function"); plt.show()

## Q6. Simple logistic regression by MLE

In [ ]:
hours = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
passed = np.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=float)

def neg_loglik_logit(beta):
    z = beta[0] + beta[1] * hours
    # stable form of  -sum[ y*z - log(1+e^z) ]
    return -np.sum(passed * z - np.logaddexp(0, z))

res = optimize.minimize(neg_loglik_logit, x0=[0.0, 0.0], method="BFGS")
b0, b1 = res.x
print("beta0 =", b0, "  beta1 =", b1)
print("Decision boundary (P = 0.5) at x =", -b0 / b1)

for h in [3, 4, 4.5, 5, 6]:
    print(f"P(pass | {h} hours) = {sigmoid(b0 + b1*h):.4f}")

**Note:** the data are *perfectly separable* (everyone with ≤ 4 hours fails, everyone with ≥ 5 hours passes). In that case the true MLE does not exist: the likelihood keeps increasing as the slope grows, so β₁ → ∞ and the optimiser just stops at a large value. The key result is the decision boundary at about 4.5 hours. Fitted probabilities are very close to 0 and 1 on either side of it.

# LAB 8 – Simulation and Monte Carlo methods

In [ ]:
rng = np.random.default_rng(2026)

## Q1. 10 000 observations from N(50, 10²)

In [ ]:
x = rng.normal(loc=50, scale=10, size=10_000)
print("Sample mean:", x.mean(), "  Sample sd:", x.std(ddof=1))

plt.hist(x, bins=50, density=True, alpha=0.6)
t = np.linspace(10, 90, 300)
plt.plot(t, stats.norm.pdf(t, 50, 10), "r")
plt.title("N(50, 10²) sample"); plt.show()

## Q2. 10 000 Bernoulli(0.3) observations and the sample proportion

In [ ]:
b = rng.binomial(n=1, p=0.3, size=10_000)
print("Sample proportion of successes:", b.mean(), "(true p = 0.3)")

## Q3. Monte Carlo estimate of ∫₀¹ x³ dx

In [ ]:
n = 1_000_000
u = rng.uniform(0, 1, n)
est = np.mean(u ** 3)
se = np.std(u ** 3, ddof=1) / np.sqrt(n)
print(f"Estimate = {est:.5f}  (SE {se:.5f});  exact value = {1/4}")

## Q4. Estimate π with one million points

In [ ]:
n = 1_000_000
px = rng.uniform(-1, 1, n)
py = rng.uniform(-1, 1, n)
inside = (px**2 + py**2) <= 1
pi_hat = 4 * inside.mean()
print("Estimate of pi:", pi_hat, "  actual:", np.pi)

The unit circle has area π and the square [−1, 1]² has area 4, so P(point inside circle) = π/4 and π ≈ 4 × (fraction inside).

## Q5. Estimate P(X > 1.96) for X ~ N(0, 1)

In [ ]:
z = rng.standard_normal(1_000_000)
p_hat = np.mean(z > 1.96)
print("Monte Carlo estimate:", p_hat)
print("Exact value         :", stats.norm.sf(1.96))   # about 0.0250

## Q6. Importance sampling for E(X²), X ~ U(0, 1)

In [ ]:
n = 100_000
# Target density f(x) = 1 on (0,1).  Proposal g(x) = 2x on (0,1)  (puts more mass near 1,
# where x^2 is large).  Sample from g by the inverse CDF: G(x)=x^2  ->  x = sqrt(U).
u = rng.uniform(0, 1, n)
xs = np.sqrt(u)
w = 1.0 / (2 * xs)              # weights f(x)/g(x)
vals = xs**2 * w                # h(x) * f(x)/g(x)

is_est = vals.mean()
is_se = vals.std(ddof=1) / np.sqrt(n)

# Plain Monte Carlo for comparison
plain = rng.uniform(0, 1, n) ** 2
print(f"Importance sampling: {is_est:.5f}  (SE {is_se:.5f})")
print(f"Plain Monte Carlo  : {plain.mean():.5f}  (SE {plain.std(ddof=1)/np.sqrt(n):.5f})")
print("Exact value        :", 1/3)

## Q7. Importance sampling vs control variates

In [ ]:
# Quick control-variate demo for the same problem: use C = X, with known E(C) = 0.5
n = 100_000
x = rng.uniform(0, 1, n)
h = x**2
c = x
cc = np.cov(h, c)
theta = cc[0, 1] / cc[1, 1]
cv_vals = h - theta * (c - 0.5)
print(f"Control variate estimate: {cv_vals.mean():.5f}  (SE {cv_vals.std(ddof=1)/np.sqrt(n):.5f})")
print(f"Plain MC estimate       : {h.mean():.5f}  (SE {h.std(ddof=1)/np.sqrt(n):.5f})")

| | **Importance sampling** | **Control variates** |
|---|---|---|
| **Idea** | Sample from a *different* distribution g that puts more weight where h(x)f(x) is large, then correct with weights f(x)/g(x). | Keep sampling from the original distribution, but subtract a correlated quantity C whose expected value is known. |
| **Estimator** | (1/n) Σ h(Xᵢ) f(Xᵢ)/g(Xᵢ), with Xᵢ ~ g | (1/n) Σ [h(Xᵢ) − θ(Cᵢ − E C)] |
| **What reduces variance** | A good choice of g (close to h·f). | High correlation between h and C. |
| **Needs** | A proposal density that is easy to sample from and covers the target. | A control variable with a *known* mean. |
| **Typical use** | Rare-event probabilities, tail estimates. | When a simpler, related quantity has an exact mean. |
| **Risk** | A poor g gives huge or infinite variance. | A weakly correlated control does little. |